In [ ]:
# The Transformer block from scratch in PyTorch, built up in three layers:
#   1. ScaledDotProductAttention - softmax(QKᵀ/√d_k) V with an optional mask.
#   2. MultiHeadAttention        - h parallel attentions in d_model/h subspaces,
#                                  concatenated and mixed by an output projection.
#   3. TransformerBlock          - pre-LN residual wrapping of attention + FFN.
# Uses only nn.Linear / nn.LayerNorm / nn.Dropout primitives (no nn.MultiheadAttention).
# See Notes/Note_4_transformer_core.md §1 (attention), §2 (heads), §4 (block), §7 (mask).

In [ ]:
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Scaled dot-product attention: the core operation every token uses to pull in a
# weighted mix of the other tokens' values. Works on any leading batch/head dims.

class ScaledDotProductAttention(nn.Module):
    """Attention(Q, K, V) = softmax(Q Kᵀ / √d_k) V.

    Shapes: q, k -> (..., n, d_k); v -> (..., n, d_v); returns (out, attn).
    mask: optional boolean tensor broadcastable to the scores (..., n_q, n_k);
          True marks positions to block (set to -inf before softmax).
    """

    def __init__(self, dropout: float = 0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        d_k = q.size(-1)
        scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)   # (..., n_q, n_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        attn = scores.softmax(dim=-1)                       # rows sum to 1
        attn = self.dropout(attn)
        out = attn @ v                                      # (..., n_q, d_v)
        return out, attn

In [ ]:
# Multi-head attention: project into h heads of width d_k = d_model/h, attend in
# parallel, concatenate, and mix across heads with W_O. Total cost ~ one full head.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention(dropout)

    def _split_heads(self, x):
        # (b, n, d_model) -> (b, h, n, d_k)
        b, n, _ = x.shape
        return x.view(b, n, self.num_heads, self.d_k).transpose(1, 2)

    def _merge_heads(self, x):
        # (b, h, n, d_k) -> (b, n, d_model)
        b, h, n, d_k = x.shape
        return x.transpose(1, 2).contiguous().view(b, n, h * d_k)

    def forward(self, query, key, value, mask=None):
        # Self-attention passes the same tensor as query/key/value; separate
        # arguments also allow cross-attention (decoder queries, encoder keys/values).
        q = self._split_heads(self.w_q(query))
        k = self._split_heads(self.w_k(key))
        v = self._split_heads(self.w_v(value))
        if mask is not None and mask.dim() == 2:
            mask = mask[None, None, :, :]   # (n, n) -> broadcast over batch & heads
        out, attn = self.attention(q, k, v, mask)
        out = self._merge_heads(out)
        return self.w_o(out), attn

In [ ]:
# Transformer block (pre-LN): each sublayer is wrapped as x + Sublayer(LayerNorm(x)).
# Attention mixes information across tokens; the position-wise FFN (GELU, 4x hidden)
# processes each token individually. Pre-LN is the stable variant for deep stacks.

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int,
                 d_ff: int | None = None, dropout: float = 0.0):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        h = self.ln1(x)
        a, attn = self.attn(h, h, h, mask)      # self-attention
        x = x + self.dropout(a)                 # residual 1
        x = x + self.dropout(self.ffn(self.ln2(x)))   # residual 2
        return x, attn

In [ ]:
# Sanity checks: shapes round-trip, attention rows are a probability distribution,
# the causal mask blocks all future positions, and our SDPA matches PyTorch's.

def causal_mask(n: int) -> torch.Tensor:
    """Boolean (n, n): True strictly above the diagonal = future positions to block."""
    return torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)


def _demo():
    torch.manual_seed(0)
    batch, seq, d_model, heads = 2, 5, 32, 4
    x = torch.randn(batch, seq, d_model)

    block = TransformerBlock(d_model, heads, dropout=0.0).eval()

    # 1. Bidirectional (encoder-style) pass: no mask, every token sees every token.
    out, attn = block(x)
    print(f"input {tuple(x.shape)} -> output {tuple(out.shape)}")
    print(f"attention weights {tuple(attn.shape)}  (batch, heads, q, k)")
    rows_ok = torch.allclose(attn.sum(-1), torch.ones(batch, heads, seq), atol=1e-6)
    print(f"attention rows sum to 1?           {rows_ok}")

    # 2. Causal (decoder-style) pass: future positions must get zero weight.
    mask = causal_mask(seq)
    _, attn_c = block(x, mask)
    future = attn_c[:, :, mask]                 # weights at the masked positions
    print(f"causal: future weights all zero?   {torch.allclose(future, torch.zeros_like(future))}")

    # 3. Cross-check ScaledDotProductAttention against torch's fused reference.
    sdpa = ScaledDotProductAttention()
    q, k, v = (torch.randn(batch, heads, seq, d_model // heads) for _ in range(3))
    ours, _ = sdpa(q, k, v)
    ref = F.scaled_dot_product_attention(q, k, v)
    print(f"matches F.scaled_dot_product_attention? {torch.allclose(ours, ref, atol=1e-6)}")

    n_params = sum(p.numel() for p in block.parameters())
    print(f"block params: {n_params:,}")


_demo()